In [4]:
!pip install xgboost


Defaulting to user installation because normal site-packages is not writeable
     ------------------------------------ 124.9/124.9 MB 268.7 kB/s eta 0:00:00


In [1]:
"""
====================================================================
DIABETES RISK PREDICTION - MODEL TRAINING & EVALUATION
Implements Research Proposal Sections:
  3.6 Data Pre-processing
  3.8 Algorithms (Logistic Regression, Random Forest, XGBoost)
  3.9 Model Performance (Accuracy, Precision, Recall, F1, AUC-ROC,
      Cross-Validation + held-out test set)
Group 17 - BDTS III - EASTC
====================================================================
"""
import pandas as pd
import numpy as np
import joblib
import json

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report,
                              confusion_matrix)
from xgboost import XGBClassifier

RANDOM_STATE = 42

In [2]:
# --------------------------------------------------------------
# 1. LOAD DATA
# --------------------------------------------------------------
df = pd.read_csv("diabetes_prediction_dataset.csv")
print(f"Raw dataset shape: {df.shape}")

# --------------------------------------------------------------
# 3.6 DATA PRE-PROCESSING
# --------------------------------------------------------------
# Remove duplicates (integrity check)
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} duplicate records")

# Keep only Male/Female for a clean binary gender encoding (proposal: encoding techniques)
df = df[df["gender"].isin(["Male", "Female"])].copy()

# Missing values: dataset has none (verified), but this guards future/real-world data
num_cols = ["age", "bmi", "HbA1c_level", "blood_glucose_level"]
df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df["smoking_history"] = df["smoking_history"].fillna(df["smoking_history"].mode()[0])

# Outlier treatment using IQR capping on continuous clinical variables (proposal 3.6)
for col in ["bmi", "blood_glucose_level"]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    df[col] = df[col].clip(lower, upper)

# Categorical encoding (proposal: label/one-hot encoding)
df["gender_male"] = (df["gender"] == "Male").astype(int)
df["smoking"] = df["smoking_history"].isin(["current", "former", "ever", "not current"]).astype(int)


Raw dataset shape: (100000, 9)
Removed 3854 duplicate records


In [3]:

# --------------------------------------------------------------
# TARGET ENGINEERING: 3-class diabetes risk label (High/Medium/Low)
# Rule grounded in ADA clinical thresholds, per proposal's stated
# dependent variable "diabetes risk level (high, medium, low)"
# --------------------------------------------------------------
def risk_level(row):
    if row["diabetes"] == 1:
        return 2  # High
    if (5.7 <= row["HbA1c_level"] <= 6.4) or (100 <= row["blood_glucose_level"] <= 125):
        return 1  # Medium (prediabetes range)
    return 0      # Low

df["risk_level"] = df.apply(risk_level, axis=1)
print("\nRisk level distribution:")
print(df["risk_level"].value_counts().rename({0: "Low", 1: "Medium", 2: "High"}))




Risk level distribution:
Low       47211
Medium    40435
High       8482
Name: risk_level, dtype: int64


In [4]:

# --------------------------------------------------------------
# FEATURE SELECTION (proposal 3.6: correlation / importance-based selection)
# --------------------------------------------------------------
FEATURES = ["age", "gender_male", "bmi", "blood_glucose_level",
            "hypertension", "heart_disease", "smoking"]
X = df[FEATURES]
y = df["risk_level"]

print("\nFeature correlation with risk_level:")
print(df[FEATURES + ["risk_level"]].corr()["risk_level"].sort_values(ascending=False))



Feature correlation with risk_level:
risk_level             1.000000
blood_glucose_level    0.211161
age                    0.182755
bmi                    0.142430
hypertension           0.134796
heart_disease          0.113541
smoking                0.064291
gender_male            0.025536
Name: risk_level, dtype: float64


In [5]:
# --------------------------------------------------------------
# TRAIN / TEST SPLIT (held-out test set, per proposal 3.9)
# --------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f"\nTrain size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}")

# Standardization (proposal 3.6: scaling techniques) - needed for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



Train size: 76902  |  Test size: 19226


In [6]:

# --------------------------------------------------------------
# 3.8 ALGORITHMS: Logistic Regression, Random Forest, XGBoost
# --------------------------------------------------------------
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced",
                                               random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=12,
                                             class_weight="balanced",
                                             random_state=RANDOM_STATE),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                              eval_metric="mlogloss", random_state=RANDOM_STATE),
}

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for name, model in models.items():
    print(f"\n{'='*60}\nTraining: {name}\n{'='*60}")

    # Logistic Regression needs scaled features; tree models do not
    Xtr = X_train_scaled if name == "Logistic Regression" else X_train
    Xte = X_test_scaled if name == "Logistic Regression" else X_test

    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    y_proba = model.predict_proba(Xte)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_test, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    auc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")

    # 5-fold cross-validation on the training set (proposal 3.9: cross-validation)
    cv_scores = cross_val_score(model, Xtr, y_train, cv=cv, scoring="accuracy", n_jobs=-1)

    results[name] = {
        "accuracy": round(acc, 4), "precision_macro": round(prec, 4),
        "recall_macro": round(rec, 4), "f1_macro": round(f1, 4),
        "auc_roc": round(auc, 4),
        "cv_accuracy_mean": round(cv_scores.mean(), 4),
        "cv_accuracy_std": round(cv_scores.std(), 4),
    }

    print(f"Accuracy:        {acc:.4f}")
    print(f"Precision(macro):{prec:.4f}")
    print(f"Recall(macro):   {rec:.4f}")
    print(f"F1(macro):       {f1:.4f}")
    print(f"AUC-ROC (ovr):   {auc:.4f}")
    print(f"5-fold CV acc:   {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    print("\nPer-class report:")
    print(classification_report(y_test, y_pred, target_names=["Low", "Medium", "High"], zero_division=0))
    print("Confusion matrix (rows=actual, cols=predicted) [Low, Medium, High]:")
    print(confusion_matrix(y_test, y_pred))

    joblib.dump(model, f"{name.lower().replace(' ', '_')}_model.joblib")

joblib.dump(scaler, "scaler_for_logistic_regression.joblib")



Training: Logistic Regression
Accuracy:        0.4811
Precision(macro):0.4590
Recall(macro):   0.5696
F1(macro):       0.4724
AUC-ROC (ovr):   0.6873
5-fold CV acc:   0.4817 (+/- 0.0025)

Per-class report:
              precision    recall  f1-score   support

         Low       0.57      0.42      0.48      9442
      Medium       0.51      0.48      0.49      8087
        High       0.30      0.80      0.44      1697

    accuracy                           0.48     19226
   macro avg       0.46      0.57      0.47     19226
weighted avg       0.52      0.48      0.48     19226

Confusion matrix (rows=actual, cols=predicted) [Low, Medium, High]:
[[3999 3622 1821]
 [2849 3885 1353]
 [ 202  129 1366]]

Training: Random Forest
Accuracy:        0.5403
Precision(macro):0.5502
Recall(macro):   0.6006
F1(macro):       0.4880
AUC-ROC (ovr):   0.7335
5-fold CV acc:   0.5461 (+/- 0.0027)

Per-class report:
              precision    recall  f1-score   support

         Low       0.57      0.77

['scaler_for_logistic_regression.joblib']

In [7]:

# --------------------------------------------------------------
# FEATURE IMPORTANCE (proposal 3.8: post-training importance analysis)
# --------------------------------------------------------------
best_name = max(results, key=lambda k: results[k]["f1_macro"])
best_model = models[best_name]
print(f"\n{'='*60}\nBEST MODEL (by macro-F1): {best_name}\n{'='*60}")

if hasattr(best_model, "feature_importances_"):
    importances = dict(zip(FEATURES, best_model.feature_importances_.round(4).tolist()))
    importances = dict(sorted(importances.items(), key=lambda x: -x[1]))
    print("Feature importances:", json.dumps(importances, indent=2))
    with open("feature_importance.json", "w") as f:
        json.dump(importances, f, indent=2)



BEST MODEL (by macro-F1): XGBoost
Feature importances: {
  "blood_glucose_level": 0.6333000063896179,
  "hypertension": 0.10890000313520432,
  "age": 0.08889999985694885,
  "heart_disease": 0.07840000092983246,
  "bmi": 0.03909999877214432,
  "gender_male": 0.02930000051856041,
  "smoking": 0.022099999710917473
}


In [8]:

# --------------------------------------------------------------
# SUMMARY TABLE + SAVE RESULTS
# --------------------------------------------------------------
summary = pd.DataFrame(results).T
summary = summary.sort_values("f1_macro", ascending=False)
print("\n===== MODEL COMPARISON SUMMARY =====")
print(summary)

with open("training_results.json", "w") as f:
    json.dump(results, f, indent=2)

summary.to_csv("model_comparison_summary.csv")
print("\nSaved: *_model.joblib, scaler_for_logistic_regression.joblib,")
print("feature_importance.json, training_results.json, model_comparison_summary.csv")  


===== MODEL COMPARISON SUMMARY =====
                     accuracy  precision_macro  recall_macro  f1_macro  \
XGBoost                0.5942           0.7343        0.5474    0.5385   
Random Forest          0.5403           0.5502        0.6006    0.4880   
Logistic Regression    0.4811           0.4590        0.5696    0.4724   

                     auc_roc  cv_accuracy_mean  cv_accuracy_std  
XGBoost               0.7338            0.5953           0.0014  
Random Forest         0.7335            0.5461           0.0027  
Logistic Regression   0.6873            0.4817           0.0025  

Saved: *_model.joblib, scaler_for_logistic_regression.joblib,
feature_importance.json, training_results.json, model_comparison_summary.csv
